In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor

In [3]:
df = pd.read_csv("data/train-test.csv")
df["date"] = pd.to_datetime(df["date"])

print("Loaded rows:", len(df))


df.loc[df["weight"] <= 0, "weight"] = np.nan

print("Missing weights after cleaning:", df["weight"].isna().sum())

Loaded rows: 48000
Missing weights after cleaning: 592


In [4]:
# training and validation sets
train = df[df["date"] < "2025-09-01"].copy()
valid = df[df["date"] >= "2025-09-01"].copy()

print("Training rows:", len(train))
print("Validation rows:", len(valid))

Training rows: 38477
Validation rows: 9523


In [6]:
#  adds the month number and weekday number. Weekdays run from 0 for Monday to 6 for Sunday
for frame in [train, valid]:
    frame["month"] = frame["date"].dt.month
    frame["day_of_week"] = frame["date"].dt.dayofweek

print(train[["date", "month", "day_of_week"]].head())

feature_columns = [
    "pickup", "delivery",
    "pickup_lat", "pickup_lon",
    "delivery_lat", "delivery_lon",
    "distance", "equipment", "weight",
    "market_index", "quote_signal",
    "month", "day_of_week"
]

        date  month  day_of_week
0 2025-01-01      1            2
1 2025-01-01      1            2
2 2025-01-01      1            2
3 2025-01-01      1            2
4 2025-01-01      1            2


In [7]:
X_train = train[feature_columns]
y_train = train["posted_rate"]

X_valid = valid[feature_columns]
y_valid = valid["posted_rate"]

categorical_columns = ["pickup", "delivery", "equipment"]

print("Training input shape:", X_train.shape)
print("Validation input shape:", X_valid.shape)

Training input shape: (38477, 13)
Validation input shape: (9523, 13)


In [8]:
model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    random_seed=42,
    allow_writing_files=False
)

model.fit(
    X_train,
    y_train,
    cat_features=categorical_columns,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=50,
    verbose=100
)

predictions = model.predict(X_valid)
mae = np.abs(y_valid - predictions).mean()

print("Model validation MAE:", round(mae, 2))
print("Trees used:", model.tree_count_)

0:	learn: 1074.5180815	test: 1099.9294282	best: 1099.9294282 (0)	total: 65.1ms	remaining: 32.5s
100:	learn: 109.8153011	test: 130.2747255	best: 130.2747255 (100)	total: 644ms	remaining: 2.54s
200:	learn: 93.6614971	test: 116.9386752	best: 116.9386752 (200)	total: 1.23s	remaining: 1.83s
300:	learn: 88.4478171	test: 115.4895282	best: 115.2647837 (291)	total: 1.82s	remaining: 1.2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 115.2647837
bestIteration = 291

Shrink model to first 292 iterations.
Model validation MAE: 115.26
Trees used: 292


In [9]:
validation_results = valid[["date", "posted_rate"]].copy()
validation_results["absolute_error"] = np.abs(y_valid - predictions)

monthly_errors = validation_results.groupby(
    validation_results["date"].dt.to_period("M")
)["absolute_error"].agg(["mean", "median", "count"])

print(monthly_errors)

               mean     median  count
date                                 
2025-09  111.648111  40.498608   4670
2025-10  118.745081  43.219022   4853


In [10]:
errors = y_valid - predictions

rmse = np.sqrt(np.mean(errors ** 2))
median_error = np.median(np.abs(errors))

print("Validation MAE:", round(mae, 2))
print("Validation RMSE:", round(rmse, 2))
print("Median absolute error:", round(median_error, 2))

Validation MAE: 115.26
Validation RMSE: 635.96
Median absolute error: 41.76


In [11]:
error_review = valid[
    ["load_id", "date", "distance", "equipment", "weight", "posted_rate"]
].copy()

error_review["predicted_rate"] = predictions
error_review["absolute_error"] = np.abs(y_valid - predictions)

error_review.nlargest(10, "absolute_error")

,load_id,date,distance,equipment,weight,posted_rate,predicted_rate,absolute_error
40096,TR-040097,2025-09-11,1760.3,Reefer,29521.0,20132.37,3816.414071,16315.955929
40171,TR-040172,2025-09-11,2283.4,Dry Van,32447.0,20361.89,4289.909308,16071.980692
43539,TR-043540,2025-10-03,1893.0,Dry Van,30760.0,17514.11,3592.305239,13921.804761
46213,TR-046214,2025-10-20,2023.4,Flatbed,34710.0,17893.17,4143.107044,13750.062956
47298,TR-047299,2025-10-27,2979.1,Reefer,30585.0,19110.30,5873.582040,13236.717960
40322,TR-040323,2025-09-12,1639.9,Dry Van,38363.0,14949.25,3305.088131,11644.161869
46516,TR-046517,2025-10-22,1684.3,Reefer,31536.0,14784.38,3545.019372,11239.360628
42881,TR-042882,2025-09-29,1460.5,Dry Van,43567.0,13503.63,2891.260796,10612.369204
42149,TR-042150,2025-09-24,1415.5,Dry Van,35333.0,13026.22,2828.133308,10198.086692
46366,TR-046367,2025-10-21,2251.9,Dry Van,40729.0,14403.14,4226.557860,10176.582140


In [12]:
rmse_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    random_seed=42,
    allow_writing_files=False
)

rmse_model.fit(
    X_train,
    y_train,
    cat_features=categorical_columns,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=50,
    verbose=100
)

rmse_predictions = rmse_model.predict(X_valid)

print("Original model MAE:", round(mae, 2))
print("Original model RMSE:", round(rmse, 2))

print(
    "New model MAE:",
    round(np.abs(y_valid - rmse_predictions).mean(), 2)
)
print(
    "New model RMSE:",
    round(np.sqrt(np.mean((y_valid - rmse_predictions) ** 2)), 2)
)

0:	learn: 1105.5175381	test: 1127.3394676	best: 1127.3394676 (0)	total: 22.6ms	remaining: 11.3s
100:	learn: 113.3265507	test: 116.5376550	best: 116.5376550 (100)	total: 455ms	remaining: 1.79s
200:	learn: 105.4824995	test: 109.5241921	best: 109.5241921 (200)	total: 851ms	remaining: 1.27s
300:	learn: 103.9458914	test: 108.5289193	best: 108.4984139 (289)	total: 1.2s	remaining: 795ms
Stopped by overfitting detector  (50 iterations wait)

bestTest = 108.4370525
bestIteration = 315

Shrink model to first 316 iterations.
Original model MAE: 115.26
Original model RMSE: 635.96
New model MAE: 108.44
New model RMSE: 632.92


In [13]:
comparison = pd.DataFrame({
    "Model": [
        "Distance baseline",
        "Equipment baseline",
        "CatBoost MAE loss",
        "CatBoost RMSE loss"
    ],
    "Validation MAE": [
        256.95,
        229.10,
        mae,
        np.abs(y_valid - rmse_predictions).mean()
    ],
    "Validation RMSE": [
        np.nan,
        np.nan,
        rmse,
        np.sqrt(np.mean((y_valid - rmse_predictions) ** 2))
    ]
})

comparison.to_csv("model_comparison.csv", index=False)
comparison

,Model,Validation MAE,Validation RMSE
0,Distance baseline,256.950000,NaN
1,Equipment baseline,229.100000,NaN
2,CatBoost MAE loss,115.264786,635.961251
3,CatBoost RMSE loss,108.437054,632.919250


In [14]:
final_inputs = pd.read_csv("data/validation.csv")
december_inputs = pd.read_csv("data/december-chart-inputs.csv")

print("Final prediction inputs:", final_inputs.shape)
print("Columns:", final_inputs.columns.tolist())

print("\nDecember chart inputs:", december_inputs.shape)
print("Columns:", december_inputs.columns.tolist())

print("\nDecember sample:")
print(december_inputs.head().to_string(index=False))

Final prediction inputs: (12000, 13)
Columns: ['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal']

December chart inputs: (31, 7)
Columns: ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']

December sample:
   pickup   delivery  distance equipment  weight       date  predicted_rate
Lexington Fort Wayne       360   Dry Van   32000 2025-12-01             NaN
Lexington Fort Wayne       360   Dry Van   32000 2025-12-02             NaN
Lexington Fort Wayne       360   Dry Van   32000 2025-12-03             NaN
Lexington Fort Wayne       360   Dry Van   32000 2025-12-04             NaN
Lexington Fort Wayne       360   Dry Van   32000 2025-12-05             NaN


In [15]:
chart_features = [
    "pickup", "delivery", "distance",
    "equipment", "weight", "month", "day_of_week"
]

chart_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    random_seed=42,
    allow_writing_files=False
)

chart_model.fit(
    train[chart_features],
    y_train,
    cat_features=categorical_columns,
    eval_set=(valid[chart_features], y_valid),
    early_stopping_rounds=50,
    verbose=100
)

chart_predictions = chart_model.predict(valid[chart_features])

print(
    "Chart model validation MAE:",
    round(np.abs(y_valid - chart_predictions).mean(), 2)
)
print("Chart model trees:", chart_model.tree_count_)

0:	learn: 1104.1673244	test: 1126.2267351	best: 1126.2267351 (0)	total: 5.6ms	remaining: 2.8s
100:	learn: 123.3898901	test: 127.1488975	best: 127.1488975 (100)	total: 413ms	remaining: 1.63s
200:	learn: 116.1201602	test: 120.3521601	best: 120.3521601 (200)	total: 787ms	remaining: 1.17s
300:	learn: 114.6038940	test: 120.4232367	best: 120.1012930 (271)	total: 1.12s	remaining: 738ms
Stopped by overfitting detector  (50 iterations wait)

bestTest = 120.101293
bestIteration = 271

Shrink model to first 272 iterations.
Chart model validation MAE: 120.1
Chart model trees: 272


In [16]:
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek

final_model = CatBoostRegressor(
    iterations=rmse_model.tree_count_,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    allow_writing_files=False
)

final_model.fit(
    df[feature_columns],
    df["posted_rate"],
    cat_features=categorical_columns,
    verbose=100
)

print("Final model trained on:", len(df), "loads")

0:	learn: 1428.7480496	total: 23.8ms	remaining: 7.48s
100:	learn: 589.9536608	total: 490ms	remaining: 1.04s
200:	learn: 583.7060370	total: 930ms	remaining: 532ms
300:	learn: 578.5065827	total: 1.33s	remaining: 66.2ms
315:	learn: 578.0635684	total: 1.39s	remaining: 0us
Final model trained on: 48000 loads


In [17]:
final_inputs["date"] = pd.to_datetime(final_inputs["date"])

final_inputs.loc[
    final_inputs["weight"] <= 0, "weight"
] = np.nan

final_inputs["month"] = final_inputs["date"].dt.month
final_inputs["day_of_week"] = final_inputs["date"].dt.dayofweek

final_rates = final_model.predict(final_inputs[feature_columns])

print("Predictions generated:", len(final_rates))
print("All finite:", np.isfinite(final_rates).all())
print("All positive:", (final_rates > 0).all())
print("First five predictions:", final_rates[:5])

Predictions generated: 12000
All finite: True
All positive: True
First five predictions: [ 856.27046624 5053.47548803 5337.27797808 4053.01525493 1797.44829989]


In [18]:
template = pd.read_csv("data/validation-predictions-template.csv")

if not final_inputs["load_id"].is_unique:
    raise ValueError("Duplicate load IDs in prediction inputs")

if not template["load_id"].is_unique:
    raise ValueError("Duplicate load IDs in template")

if set(template["load_id"]) != set(final_inputs["load_id"]):
    raise ValueError("Template IDs do not match prediction inputs")

rates_by_id = pd.Series(
    final_rates,
    index=final_inputs["load_id"]
)

template["predicted_rate"] = template["load_id"].map(rates_by_id)

template.to_csv(
    "validation_predictions.csv",
    index=False,
    float_format="%.2f"
)

print("Saved validation_predictions.csv")
print("Rows:", len(template))
print("Missing predictions:", template["predicted_rate"].isna().sum())


Saved validation_predictions.csv
Rows: 12000
Missing predictions: 0


In [19]:
final_chart_model = CatBoostRegressor(
    iterations=chart_model.tree_count_,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    allow_writing_files=False
)

final_chart_model.fit(
    df[chart_features],
    df["posted_rate"],
    cat_features=categorical_columns,
    verbose=100
)

print("Final chart model trained on:", len(df), "loads")

0:	learn: 1427.7976868	total: 7.54ms	remaining: 2.04s
100:	learn: 592.4278045	total: 482ms	remaining: 816ms
200:	learn: 587.6982990	total: 865ms	remaining: 306ms
271:	learn: 584.3036988	total: 1.14s	remaining: 0us
Final chart model trained on: 48000 loads


In [20]:
december_inputs["date"] = pd.to_datetime(december_inputs["date"])
december_inputs["month"] = december_inputs["date"].dt.month
december_inputs["day_of_week"] = december_inputs["date"].dt.dayofweek

december_rates = final_chart_model.predict(
    december_inputs[chart_features]
)

if not np.isfinite(december_rates).all() or (december_rates <= 0).any():
    raise ValueError("December predictions must be finite and positive")

december_output = december_inputs[
    ["pickup", "delivery", "distance", "equipment",
     "weight", "date", "predicted_rate"]
].copy()

december_output["predicted_rate"] = december_rates

december_output.to_csv(
    "december_predictions.csv",
    index=False,
    date_format="%Y-%m-%d",
    float_format="%.2f"
)

print("Saved december_predictions.csv")
print("Rows:", len(december_output))
print(december_output.head())

Saved december_predictions.csv
Rows: 31
      pickup    delivery  distance equipment  weight       date  \
0  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-01   
1  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-02   
2  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-03   
3  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-04   
4  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-05   

   predicted_rate  
0      862.621755  
1      876.521527  
2      898.155303  
3      897.291020  
4      889.429955  
